# ARIMA on a synthetic jam: what autocorrelation can and cannot tell you

A natural idea for a traffic benchmark: *the jam has a rhythm - rush hour every day, a weekly pattern - so fit an ARIMA / SARIMAX, an AR(7), an AR on 14 or 28 days, a daily × weekly seasonal model, and forecast the jam from its own past.*
This notebook tests that **properly** - on 10 corridors × 273 train days, with a null distribution made from the same data with the days shuffled - and reports where it holds and where it does not.

**What the data say (live cells below; the summary of the larger run in our log is in section 7)**
* **The jam has almost no memory from day to day.** A weekday profile explains 4.7 % of the variance of the daily jam length per corridor (2.2 % by chance; significant on 5 of 10 corridors; 6.4 % on the corridor mean, p = 0.007). After removing it, the autocorrelation at 7, 14, 21, 28 days is −0.03…−0.01. No AR or SARIMAX model beats *'tomorrow = the training mean'* (median RMSE ratio 0.997 - 1.009).
* **The flow does have a weekly cycle**, and it is the *profile*, not a memory: weekday profile η² 0.26 per corridor, 0.72 on the corridor mean; the profile scores 0.905 against the constant, the best ARMA-type model 0.979. The tell-tale: ACF peaks at 7, 14, 21, 28 days of **equal height** (≈ 0.23) - a decaying memory would fall off - and they are all gone after removing the profile.
* The 'weak pattern after differencing' (lag-1 ACF ≈ −0.45) is the **−0.5 of over-differencing** - white noise gives −0.5.
* Within a day, memory of the jam is a matter of **hours** (residual ACF after the hour-of-day profile: 0.75 at 1 h, 0.14 at 6 h, 0.03 at 12 h). That is where Task 2's 30-minute horizon lives - and there the simplest forecast is already hard to beat (section 7).

Everything here is the *released* train data (observed cells only); nothing is tuned.

In [ ]:
import os, glob, numpy as np, pandas as pd
from pathlib import Path
KIN = Path(os.environ.get('KIN', '/kaggle/input'))
root = [p for depth in range(1, 6) for p in KIN.glob('/'.join(['*'] * depth) + '/corridors') if (p / 'D12_I5_N').exists()][0]
def corridor(panel):
    """links in driving order, x_km along the direction of travel, free-flow speed and length per link"""
    topo = pd.read_csv(f'{root}/{panel}/network/lwr_mainline_topology.csv'); fd = pd.read_csv(f'{root}/{panel}/network/fd_parameters.csv').set_index('link_id')
    f = sorted(glob.glob(f'{root}/{panel}/train/mainline_states/*/synthetic_mainline_*.parquet'))[0]
    mp = pd.read_parquet(f, columns=['link_id', 'milepost']).groupby('link_id').milepost.mean(); ml = set(topo.link_id)
    s = [np.sign(mp[o] - mp[r.link_id]) for r in topo.itertuples() for o in str(r.outgoing_link_ids).split(';') if o in ml and r.link_id in mp.index and o in mp.index]
    sgn = 1.0 if np.sum(s) >= 0 else -1.0
    order = (mp[mp.index.isin(ml)] * sgn).sort_values().index.tolist(); x = mp.reindex(order).to_numpy() * sgn
    return dict(order=order, x=x - x.min(), vf=fd.loc[order, 'free_speed_kmh'].to_numpy(), length=fd.loc[order, 'length_km'].to_numpy(), cap=fd.loc[order, 'capacity_vph'].to_numpy(), lanes=fd.loc[order, 'lanes'].to_numpy())
def day_grid(panel, date, order, col='speed_kmh'):
    """(288, L) grid of one train day; NaN where the cell is not score-eligible (not observed)"""
    f = glob.glob(f'{root}/{panel}/train/mainline_states/*/synthetic_mainline_{date.replace("-", "_")}.parquet')[0]
    d = pd.read_parquet(f, columns=['timestamp', 'link_id', col, 'is_score_eligible']); d = d[d.is_score_eligible.astype(bool)]
    li = {l: i for i, l in enumerate(order)}; t = pd.to_datetime(d.timestamp); G = np.full((288, len(order)), np.nan)
    G[(t.dt.hour * 12 + t.dt.minute // 5).to_numpy(), d.link_id.map(li).to_numpy()] = d[col].to_numpy(); return G
import matplotlib.pyplot as plt
print('competition data at', root)

## 1. Three daily series per corridor

* **jam length** (km): per link the share of its observed 5-minute cells with speed < 0.6 × free-flow speed (the competition's own queue definition), times the link length, summed over the corridor = the average length of road in jam during the day;
* **flow**: observed flow / link capacity, averaged over links;
* and the same jam share per **hour** (for section 6).

Only observed (score-eligible) cells are used, so nothing in the series is imputed.

In [ ]:
# data: per corridor and day, two jam measures and the flow, from the competition's train layer (observed cells only)
import glob, time
import matplotlib.pyplot as plt
from scipy import stats
panels = sorted(p.name for p in Path(root).iterdir() if p.is_dir() and (p / 'train').exists()); daily, hourly = [], []; t0 = time.time()
for panel in panels:
    c = corridor(panel); vfm, capm, lenm = (dict(zip(c['order'], c[k])) for k in ('vf', 'cap', 'length'))
    for f in sorted(glob.glob(f'{root}/{panel}/train/mainline_states/*/synthetic_mainline_*.parquet')):
        d = pd.read_parquet(f, columns=['date', 'timestamp', 'link_id', 'speed_kmh', 'flow_vph', 'is_score_eligible']); d = d[d.is_score_eligible.astype(bool)]
        d['jam'] = d.speed_kmh.to_numpy() < 0.6 * d.link_id.map(vfm).to_numpy(); d['q'] = d.flow_vph / d.link_id.map(capm).to_numpy(); d['hour'] = pd.to_datetime(d.timestamp).dt.hour
        fl = d.groupby('link_id').jam.mean(); jam_km = float((fl * fl.index.map(lenm)).sum()); date = d.date.iloc[0]
        daily.append((panel, date, jam_km, float(d.groupby('link_id').q.mean().mean()))); h = d.groupby('hour').jam.mean().reindex(range(24)); hourly.append((panel, date, *h.to_numpy()))
D = pd.DataFrame(daily, columns=['panel', 'date', 'jam_km', 'flow']); D['date'] = pd.to_datetime(D.date); D['dow'] = D.date.dt.dayofweek
Hh = pd.DataFrame(hourly, columns=['panel', 'date'] + list(range(24))); Hh['date'] = pd.to_datetime(Hh.date)
print(f'{len(panels)} corridors x {D.groupby("panel").size().iloc[0]} train days, {time.time() - t0:.0f} s;  mean jam length {D.jam_km.mean():.1f} km/day, mean flow {D.flow.mean():.2f} of capacity')
D.groupby('panel')[['jam_km', 'flow']].agg(['mean', 'std']).round(2)

## 2. Is there a weekly rhythm? (η² = share of the variance a weekday profile explains)

*By chance* = the same statistic after shuffling the order of the days (300 times): with 273 days and 7 groups it is never zero. The last two rows average the 10 corridors first: independent daily noise cancels, a shared weekly pattern does not.

In [ ]:
# weekday effect: share of the variance a weekday profile explains, tested against shuffled days
rng = np.random.default_rng(0); out = []
for m in ('jam_km', 'flow'):
    eta, eta0, pv = [], [], []
    for p, g in D.groupby('panel'):
        y = g[m].to_numpy(); dow = g.dow.to_numpy(); pv.append(stats.f_oneway(*[y[dow == k] for k in range(7)]).pvalue)
        eta.append(1 - sum(((y[dow == k] - y[dow == k].mean()) ** 2).sum() for k in range(7)) / ((y - y.mean()) ** 2).sum())
        e0 = []
        for _ in range(300):
            ys = rng.permutation(y); e0.append(1 - sum(((ys[dow == k] - ys[dow == k].mean()) ** 2).sum() for k in range(7)) / ((ys - ys.mean()) ** 2).sum())
        eta0.append(np.mean(e0))
    out.append(dict(measure=m, eta2_mean=np.mean(eta), eta2_by_chance=np.mean(eta0), corridors_p_lt_005=f'{sum(p < 0.05 for p in pv)} of {len(pv)}', pooled_p=stats.combine_pvalues(pv, method='fisher')[1]))
for m in ('jam_km', 'flow'):                                 # the same on the corridor-mean series: independent noise of 10 corridors averages out, the shared weekly pattern stays
    g = D.groupby('date')[[m]].mean().reset_index(); g['dow'] = g.date.dt.dayofweek; y = g[m].to_numpy(); dow = g.dow.to_numpy(); ss = lambda v: 1 - sum(((v[dow == k] - v[dow == k].mean()) ** 2).sum() for k in range(7)) / ((v - v.mean()) ** 2).sum()
    out.append(dict(measure=m + ' (mean of 10 corridors)', eta2_mean=ss(y), eta2_by_chance=np.mean([ss(rng.permutation(y)) for _ in range(300)]), corridors_p_lt_005='-', pooled_p=stats.f_oneway(*[y[dow == k] for k in range(7)]).pvalue))
pd.DataFrame(out).set_index('measure').round(4)

Read it as: the **flow** has a strong weekly profile (commuters), the **jam length** a weak one that is real but tiny (6 % on the corridor mean; it was easy to miss with an ACF alone - the lag-7 autocorrelation of the jam length is ≈ 0.01 - and shows only in the F-test). Our own log first called it 'no weekly pattern' and had to correct that: the weekly effect in the jam exists, and is useless for forecasting (below).

## 3. Autocorrelation per day, raw and after removing the weekday profile

Grey band: what the ACF does when the days are *independent* (shuffled), averaged over the 10 corridors like the bars.

In [ ]:
# ACF per day: raw, after removing the weekday profile, and the band you get when the days are independent (shuffled)
def acf(y, nl):
    y = y - y.mean(); d = (y * y).sum(); return np.array([(y[:-k] * y[k:]).sum() / d for k in range(1, nl + 1)])
def deweek(g, m): return (g[m] - g.groupby('dow')[m].transform('mean')).to_numpy()
NL = 28; fig, axs = plt.subplots(1, 2, figsize=(13, 3.8)); res = {}
for ax, m, title in zip(axs, ('jam_km', 'flow'), ('length of the jam (km, daily mean)', 'flow (share of capacity, daily mean)')):
    raw = np.mean([acf(g[m].to_numpy(), NL) for _, g in D.groupby('panel')], 0); dw = np.mean([acf(deweek(g, m), NL) for _, g in D.groupby('panel')], 0)
    null = np.array([np.mean([acf(rng.permutation(g[m].to_numpy()), NL) for _, g in D.groupby('panel')], 0) for _ in range(300)]); lo, hi = np.percentile(null, [2.5, 97.5], axis=0); res[m] = (raw, dw, lo, hi)
    x = np.arange(1, NL + 1); ax.fill_between(x, lo, hi, color='#cfd3d8', label='95 % band if the days were independent (10 corridors averaged)'); ax.bar(x - 0.2, raw, 0.4, color='#2a6fb0', label='raw'); ax.bar(x + 0.2, dw, 0.4, color='#b3261e', label='after removing the weekday profile')
    for k in (7, 14, 21, 28): ax.axvline(k, color='k', lw=0.4, ls=':')
    ax.set_title(title, fontsize=10); ax.set_xlabel('lag (days)'); ax.set_ylabel('autocorrelation'); ax.legend(fontsize=7, frameon=False, loc='upper right')
plt.tight_layout(); plt.show()
print('outside the band, lags 1..28:   ' + '   '.join(f'{m}: raw {int(((r < lo) | (r > hi)).sum())}, after weekday profile {int(((d_ < lo) | (d_ > hi)).sum())}' for m, (r, d_, lo, hi) in res.items()))
print('lags outside the band, raw -> after the weekday profile:   ' + '   '.join(f'{m}: {[int(k) for k in np.flatnonzero((r < lo) | (r > hi)) + 1]} -> {[int(k) for k in np.flatnonzero((d_ < lo) | (d_ > hi)) + 1]}' for m, (r, d_, lo, hi) in res.items()))
print('mean ACF at lags 7/14/21/28:    ' + '   '.join(f'{m}: raw {np.round(r[[6, 13, 20, 27]], 2)}, after {np.round(d_[[6, 13, 20, 27]], 2)}' for m, (r, d_, lo, hi) in res.items()))

* **Flow:** 27 of the 28 raw lags are outside the band and the peaks at 7, 14, 21, 28 days stand at the same height. That is the signature of a **deterministic cycle**: an AR process decays with the lag, a repeating profile does not. After subtracting the weekday profile, 0 of 28 lags are outside the band.
* **Jam length:** the raw ACF is tiny (|r| ≤ 0.05), but the bars at 14, 21 and 28 days touch or just pass the band - the faint weekly effect of section 2 - and after removing the weekday profile they are at −0.01. Three bars are outside the band after the profile too (lags printed above): 3 of 28 is a little more than the ~1.4 expected from a 95 % band, and none of them is a multiple of 7.

## 4. The 'weak pattern after differencing' is an artefact

Differencing a series with no memory at all turns white noise into a series with lag-1 autocorrelation **−0.5** (each value appears in two differences with opposite sign).

In [ ]:
# the 'weak pattern after differencing' is an artefact: first differences of ANY memory-less series have lag-1 autocorrelation -0.5
dl = [np.mean([acf(np.diff(g[m].to_numpy()), 1)[0] for _, g in D.groupby('panel')]) for m in ('jam_km', 'flow')]
dls = [np.mean([acf(np.diff(rng.permutation(g[m].to_numpy())), 1)[0] for _, g in D.groupby('panel')]) for m in ('jam_km', 'flow')]
print(f'lag-1 ACF of the first differences   jam_km: data {dl[0]:+.3f}, days shuffled {dls[0]:+.3f}   flow: data {dl[1]:+.3f}, days shuffled {dls[1]:+.3f}   white noise: {acf(np.diff(rng.normal(size=100000)), 1)[0]:+.3f}')

The data, the same data with the days shuffled, and plain white noise all land on −0.43…−0.50. A moderately negative lag-1 ACF in a differenced series is not evidence of anything.

## 5. Does any ARIMA / SARIMAX beat 'tomorrow looks like the mean'?

Fit on the first 200 days, **one-step-ahead** forecast of each of the last 73 days (the model sees the actual past each time), each series standardised with the training mean and sd. Score = RMSE divided by the RMSE of the constant (training mean).
`weekday_profile` is the plain table of 7 means. 'n better' counts corridors with a ratio below 0.98.

In [ ]:
# does any ARIMA/SARIMAX beat 'tomorrow looks like the mean'? one-step-ahead, fit on the first 200 days, scored on the last 73
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings; warnings.simplefilter('ignore')          # statsmodels re-enables its convergence warnings on import
SPECS = {'AR(1)': dict(order=(1, 0, 0)), 'AR(1,7)': dict(order=([1, 7], 0, 0)), 'seasonal AR(7)': dict(order=(1, 0, 0), seasonal_order=(1, 0, 0, 7)), 'ARIMA(0,1,1)': dict(order=(0, 1, 1)), 'AR(1,7,14,28)': dict(order=([1, 7, 14, 28], 0, 0))}
def one_step(y, spec, ntr=200):
    r = SARIMAX(y[:ntr], trend='c' if spec['order'][1] == 0 else None, **spec).fit(disp=False); ap = r.apply(y, refit=False); return ap.get_prediction(start=ntr, dynamic=False).predicted_mean
tab = []
for m in ('jam_km', 'flow'):
    for p, g in D.groupby('panel'):
        y = g[m].to_numpy(); y = (y - y[:200].mean()) / y[:200].std(); te = y[200:]; wk = g.assign(y=y).iloc[:200].groupby('dow').y.mean(); prof = g.dow.iloc[200:].map(wk).to_numpy()
        row = dict(measure=m, panel=p, constant=np.sqrt(np.mean((te - 0) ** 2)), weekday_profile=np.sqrt(np.mean((te - prof) ** 2)))
        for nm, sp in SPECS.items():
            try: row[nm] = np.sqrt(np.mean((te - np.asarray(one_step(y, sp))) ** 2))
            except Exception: row[nm] = np.nan
        tab.append(row)
T = pd.DataFrame(tab); ratio = T.set_index(['measure', 'panel']).div(T.set_index(['measure', 'panel'])['constant'], axis=0).drop(columns='constant')
summ = ratio.groupby('measure').median().T.round(3); summ['corridors better than constant, jam_km'] = (ratio.loc['jam_km'] < 0.98).sum().drop('constant', errors='ignore'); summ['corridors better than constant, flow'] = (ratio.loc['flow'] < 0.98).sum().drop('constant', errors='ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 20)
print('median RMSE ratio to the constant (training mean), 10 corridors; <1 is better; "better" = ratio < 0.98')
summ.rename(columns={'corridors better than constant, jam_km': 'n better (jam_km)', 'corridors better than constant, flow': 'n better (flow)'})

* **Jam length: nothing beats the constant** (0.997 - 1.009; the weekday profile is *worse*, 1.006: it is real in section 2, but its seven means are estimated from only ~29 days each, and on 73 test days that costs about as much as it saves - a plausible reason, not tested here).
* **Flow:** the weekday profile is the winner (0.905, better on 9 of 10 corridors); AR terms with a lag at 7 get ≈ 0.98 on half of the corridors - consistent with them approximating the profile.
* One-step-ahead for a day is the *easy* case for an AR model (it sees yesterday's actual value); AR-type models revert towards the mean at longer horizons, so a model that cannot win here is not expected to win further out; only this case is shown.

## 6. Within the day: the hour-of-day profile, and what is left

Hourly jam share per corridor; the 'profile' is the mean per hour of day.

In [ ]:
# hourly: the daily rhythm is the whole story at 1 h; what is left after removing it?
fig, axs = plt.subplots(1, 2, figsize=(13, 3.6)); NLH = 168; rawh, resh, nullh = [], [], []
for p, g in Hh.groupby('panel'):
    y = g[list(range(24))].to_numpy().reshape(-1); prof = np.nanmean(g[list(range(24))].to_numpy(), 0); r = (g[list(range(24))].to_numpy() - prof).reshape(-1); rawh.append(acf(np.nan_to_num(y, nan=np.nanmean(y)), NLH)); resh.append(acf(np.nan_to_num(r), NLH))
rawh, resh = np.mean(rawh, 0), np.mean(resh, 0); x = np.arange(1, NLH + 1)
axs[0].bar(x, rawh, color=['#b3261e' if k % 24 == 0 else '#2a6fb0' for k in x]); axs[0].set_title('hourly jam share, raw ACF: peaks at 24, 48, ... h = the daily profile', fontsize=10); axs[1].bar(x, resh, color='#2a6fb0'); axs[1].set_title('after removing the hour-of-day profile', fontsize=10)
for a in axs: a.set_xlabel('lag (hours)'); a.set_ylabel('autocorrelation')
plt.tight_layout(); plt.show()
share = [1 - np.nanvar(g[list(range(24))].to_numpy() - np.nanmean(g[list(range(24))].to_numpy(), 0)) / np.nanvar(g[list(range(24))].to_numpy()) for _, g in Hh.groupby('panel')]; print(f'share of the hourly variance explained by the hour-of-day profile: mean {np.mean(share):.2f} over the 10 corridors (range {min(share):.2f}-{max(share):.2f})')
print(f'ACF of the residual at lag 1 / 3 / 6 / 12 / 24 / 48 / 168 h: ' + ' / '.join(f'{resh[k - 1]:+.3f}' for k in (1, 3, 6, 12, 24, 48, 168)))

The raw ACF has peaks at 24, 48, 72 … h: that is the daily profile (left). Once the profile is removed (right) the memory of the jam is short - hours - and nothing is left at 24, 48 or 168 h (all within ±0.05). The hour-of-day profile explains about half of the hourly variance on average, and anywhere between 11 % and 88 % depending on the corridor.

## 7. The larger run (static numbers from our measurement log, 2026-11-04)

The live cells above are a compact version. Our measurement log has the full grid (result files `jam_acf_*.json` and `t1_jam_sarimax_grid_*.csv`; these files and their scripts are not included here, so the table is a report, not something you can re-run from this notebook):

| test | result |
|---|---|
| **152 SARIMAX specs per series**, order chosen by AIC on the training days, 10 corridors, daily | jam length: median RMSE ratio to the constant **0.993** (30-day horizon) / **0.999** (one step), **0 of 10** corridors below 0.98. Flow: **0.939 / 0.933**, 6–7 of 10 corridors. Pooled over corridors, flow: 0.584 (series with shuffled days: ≈ 1.00) |
| *cherry-picking* the best spec on the test days | looks better than the AIC choice for both measures - and is not to be trusted |
| hourly, AIC-chosen (2,0,0)(1,0,1,24) | 0.958 vs the hour-of-day profile for the raw jam length, **1.004** for the series with the profile removed (no gain) |
| **double-seasonal AR** (hour AR(24) × day AR(7); direct *h*-step forecast, pooled hourly series, learned on 200 days, tested on 73) | worse than the plain weekday × hour table at every horizon for flow and q·v ('a single observation per lag is noisier than a table that averages ~39 weeks - the table *is* the weekly rhythm'); for the jam length +48 % lower MSE than the table at +1 h but **15–28 % higher** at ≥ 3 h |
| table + **short-term correction** (table + γ × the deviation from the table now) | lower MSE than the table by 89 % at +1 h, 47 % at +3 h, 10 % at +6 h, 0 % at +12 h (jam length). Lags of 24 h and 168 h on top of that: −0.7 … +1.7 % = nothing |
| ARIMA **per detector** at 5-minute resolution (454 link-windows inside jams, February, 3 corridors; fitted on the last 6 h; 6 steps ahead) | speed RMSE at +30 min: persistence **28.5**, ARIMA(0,1,1) 28.6, (1,1,0) 28.7, (1,0,0) 30.1, (2,1,1) 32.1, linear trend 42.7 km/h. ARIMA = persistence or worse |

**How to read the last three rows together.** There is a real, short-lived memory (hours) *around the profile*. A forecast = (profile for this weekday and hour) + (a decaying fraction of how far the road is from the profile right now). That is what a tree on 'profile features + last observed state' already does, and why a 30-minute forecast is dominated by persistence. The **unknown** is the *start* of a new jam - it is not in the past of the series.

Corrections we had to make to ourselves on the way (they are in the log as appended notes): our first reading of the day-level ACF ('a flat plateau', 'always slightly positive at 14-42 days') was wrong - 14 of 29 lags were negative; the positive ones were the multiples of 7, i.e. the weak weekly effect, and they vanish after the weekday profile.

## 8. Take-aways
1. **Always make the null from your own data** (shuffle the days) and plot it. The textbook band ±1.96/√n (±0.12 for 273 days) is what most ACF plots show; for these averaged series the shuffled-days band is a lot narrower (±0.03) and reads very differently.
2. **Equal-height peaks at 7, 14, 21, 28 = a deterministic cycle**, to be modelled with a profile (a table), not with AR lags.
3. **Differenced series have lag-1 ACF ≈ −0.5 whatever is in them.**
4. **Cherry-picking the best of 152 specs on the test days is a forecast of nothing.** Choose by AIC on the training data and report the median over corridors.
5. For a short horizon, 'persistence + profile' is a strong baseline; an ARIMA that does not beat it is not a negative result about ARIMA, it is a statement about how much memory this process has.

*Not measured here: a 5-minute SARIMAX with two seasonal periods (statsmodels supports one); horizons shorter than 1 h for the aggregated series; flow and speed at the level of single cells beyond the 454-window test above. Data: the competition release only; the live cells take about a minute. Written with Claude Code.*